In [0]:
%pip install boto3 -q


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
os.environ["AWS_ACCESS_KEY_ID"] = "Your Access Key "
os.environ["AWS_SECRET_ACCESS_KEY"] = "Your Secret Access Key "
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"


In [0]:
import boto3, json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, when

BUCKET = "nbapredictions-sthomas26-ncf"
SEASON = "2024-25"

s3 = boto3.client("s3")

# Find latest bronze games file
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"bronze/games/season={SEASON}/")
latest_key = sorted([o["Key"] for o in resp["Contents"]])[-1]
print(f"Reading: {latest_key}")

obj = s3.get_object(Bucket=BUCKET, Key=latest_key)
raw = json.loads(obj["Body"].read())

result_set = raw["resultSets"][0]
headers = result_set["headers"]
rows = result_set["rowSet"]
print(f"Columns: {headers}")
print(f"Rows: {len(rows)}")

Reading: bronze/games/season=2024-25/run=20260514T004017/games.json
Columns: ['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
Rows: 2802


In [0]:
spark = SparkSession.builder.appName("NBA-Silver").getOrCreate()

df = spark.createDataFrame(rows, schema=headers)

silver_df = df.select(
    col("GAME_ID"),
    col("GAME_DATE"),
    col("TEAM_ID"),
    col("TEAM_ABBREVIATION"),
    col("TEAM_NAME"),
    col("MATCHUP"),
    col("WL"),
    col("PTS").cast("int"),
    col("FG_PCT").cast("float"),
    col("FG3_PCT").cast("float"),
    col("FT_PCT").cast("float"),
    col("REB").cast("int"),
    col("AST").cast("int"),
    col("TOV").cast("int"),
    col("PLUS_MINUS").cast("float")
).dropna(subset=["GAME_ID", "TEAM_ID", "WL"])

silver_df = silver_df.withColumn(
    "GAME_DATE", to_date(col("GAME_DATE"), "yyyy-MM-dd")
)

print(f"Silver rows: {silver_df.count()}")
silver_df.show(5)

Silver rows: 2802
+----------+----------+----------+-----------------+--------------------+-----------+---+---+------+-------+------+---+---+---+----------+
|   GAME_ID| GAME_DATE|   TEAM_ID|TEAM_ABBREVIATION|           TEAM_NAME|    MATCHUP| WL|PTS|FG_PCT|FG3_PCT|FT_PCT|REB|AST|TOV|PLUS_MINUS|
+----------+----------+----------+-----------------+--------------------+-----------+---+---+------+-------+------+---+---+---+----------+
|0042400407|2025-06-22|1610612760|              OKC|Oklahoma City Thu...|OKC vs. IND|  W|103| 0.402|  0.275|  0.71| 40| 20|  7|      12.0|
|0042400407|2025-06-22|1610612754|              IND|      Indiana Pacers|  IND @ OKC|  L| 91| 0.414|  0.393| 0.759| 45| 17| 21|     -12.0|
|0042400406|2025-06-19|1610612760|              OKC|Oklahoma City Thu...|  OKC @ IND|  L| 91| 0.419|  0.267| 0.808| 41| 14| 21|     -17.0|
|0042400406|2025-06-19|1610612754|              IND|      Indiana Pacers|IND vs. OKC|  W|108| 0.413|  0.357|  0.68| 46| 23| 10|      17.0|
|00424004

In [0]:
import os
import tempfile
import pandas as pd

# Convert to pandas and create a clean copy to remove Spark metadata
pandas_df = pd.DataFrame(silver_df.toPandas())

# Use Python's tempfile (not Spark's /tmp) to avoid DBFS restriction
with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = f"{tmpdir}/silver_games.parquet"
    pandas_df.to_parquet(parquet_path, index=False)
    
    # Upload to S3
    s3.upload_file(
        parquet_path,
        BUCKET,
        f"silver/games/season={SEASON}/games.parquet"
    )
    print(f"uploaded → silver/games/season={SEASON}/games.parquet")

uploaded → silver/games/season=2024-25/games.parquet
